In [1]:
# !pip install 'zarr<3'
# !pip install timm


In [2]:
# # Install CellSAM
# # !pip install git+https://github.com/vanvalenlab/cellSAM.git
# # Alternative installation with all dependencies
# !pip install torch torchvision  # Make sure PyTorch is installed first
# !pip install git+https://github.com/vanvalenlab/cellSAM.git

In [3]:
# ALWAYS RUN THIS FIRST!
import os
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Specify GPU 0 (out of 4 available GPUs)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

NOTEBOOK_DIR = Path("/rsrch9/home/plm/idso_fa1_pathology/codes/yshokrollahi/vitamin-p-latest")
os.chdir(NOTEBOOK_DIR)
sys.path.insert(0, str(NOTEBOOK_DIR))
print(f"✅ Working directory: {os.getcwd()}")
print(f"✅ Using GPU: {os.environ.get('CUDA_VISIBLE_DEVICES', 'Not set')}")

✅ Working directory: /rsrch9/home/plm/idso_fa1_pathology/codes/yshokrollahi/vitamin-p-latest
✅ Using GPU: 0


In [4]:
# Cell 3: Import and create dataloaders
from dataset import Config, create_dataloaders

# Just use the correct relative path from your working directory
config = Config("configs/training/newmethod.yaml")  # Note: "configs" not "config"
config.print_config()

train_loader, val_loader, test_loader = create_dataloaders(config)
print("\n✅ Ready to use!")

✅ CRC Dataset Package v1.0.0 loaded
CRC DATASET CONFIGURATION
Config File: configs/training/newmethod.yaml
Zarr Base: /rsrch9/home/plm/idso_fa1_pathology/TIER2/yasin-vitaminp/cpm15
Cache: Disabled
Strategy: memory

📊 Data Splits:
  Train: 3 samples
  Val: 3 samples
  Test: 4 samples

🔄 DataLoader:
  Batch Size: 4
  Num Workers: 16
  Pin Memory: True

🎨 Augmentation:
  Training: True
  Probability: 1.0

🎯 HV Maps:
  Generate: True
  Method: pannuke
  HE Nuclei: True
  HE Cells: True
  MIF Nuclei: True
  MIF Cells: True

🔍 Filtering:
  Min Instances: 0
  Filter Empty: True

CREATING DATALOADERS
Strategy: memory
Use Cache: False
Batch Size: 4
Num Workers: 16
Train split: 0 CRC + 0 Xenium + 0 TissueNet + 0 PanNuke + 0 Lizard + 0 MoNuSeg + 0 MoNuSAC + 0 TNBC + 0 NuInsSeg + 0 CryoNuSeg + 0 BC + 0 CoNSeP + 0 Kumar + 0 CPM17 + 1 CPM15 + 1 HE_DSB2018 + 1 Fluo_DSB2018 + 0 PanopTILs
Val   split: 0 CRC + 0 Xenium + 0 TissueNet + 0 PanNuke + 0 Lizard + 0 MoNuSeg + 0 MoNuSAC + 0 TNBC + 0 NuInsSeg + 

In [5]:
# Step 1: Check what's available in cellSAM
import cellSAM
print("Available functions in cellSAM:")
print(dir(cellSAM))

Available functions in cellSAM:
['AnchorDETR', 'CellSAM', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', '_auth', 'cellsam_pipeline', 'download_training_data', 'get_local_model', 'get_model', 'model', 'sam_inference', 'segment_cellular_image', 'utils', 'wsi']


In [6]:
import torch
import numpy as np
import os
from collections import defaultdict
from tqdm import tqdm
from PIL import Image

# Import your metrics
from metrics import get_fast_pq, aggregated_jaccard_index

# CellSAM Imports
from cellSAM import get_model, segment_cellular_image

# ==========================================
# 1. CONFIGURATION
# ==========================================
DATASET_MODALITIES = {
    # MIF Only Datasets
    'fluorescence_dsb2018': {'he_nuclei': False, 'he_cell': False, 'mif_nuclei': True,  'mif_cell': False},
    
    # H&E Only Datasets
    'he_dsb2018': {'he_nuclei': True,  'he_cell': False, 'mif_nuclei': False, 'mif_cell': False},
    'cpm15':      {'he_nuclei': True,  'he_cell': False, 'mif_nuclei': False, 'mif_cell': False},
    'panoptils':      {'he_nuclei': True,  'he_cell': False, 'mif_nuclei': False, 'mif_cell': False}

}


# ==========================================
# 2. SETUP
# ==========================================
os.environ['DEEPCELL_ACCESS_TOKEN'] = 'JbVVUStF.A6Ec6pe5vKsoB3RhTnSOaqXJ1thDE3B6'

device_str = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device_str}")

print("\n📦 Loading CellSAM model...")
model = get_model(model='cellsam_general')
print("✅ Model loaded successfully!")

dataset_metrics = defaultdict(lambda: defaultdict(list))
dataset_counts = defaultdict(int)

def tensor_to_numpy_img(tensor):
    img = tensor.permute(1, 2, 0).cpu().numpy()
    # Normalize to 0-255 uint8
    if img.max() > img.min():
        img = (img - img.min()) / (img.max() - img.min())
    else:
        # Image is flat/constant
        img = np.zeros_like(img) 
        
    img = (img * 255).astype(np.uint8)
    return img

def safe_cellsam_predict(img_np, model, device):
    """
    Wraps CellSAM inference to handle library crashes on empty/difficult tiles.
    """
    # 1. Check for flat images (e.g., all black/white background tiles)
    # CellSAM's percentile normalization fails on these, causing TypeError
    if np.min(img_np) == np.max(img_np):
        return np.zeros((img_np.shape[0], img_np.shape[1]), dtype=np.int32)
        
    try:
        # 2. Run CellSAM
        mask, _, _ = segment_cellular_image(img_np, model=model, device=device)
        return mask.squeeze()
    except (AttributeError, ValueError, RuntimeError, TypeError):
        # 3. Catch TypeError specifically (for the "not a sequence" error) 
        # and others for general stability.
        return np.zeros((img_np.shape[0], img_np.shape[1]), dtype=np.int32)

# ==========================================
# 3. EVALUATION LOOP
# ==========================================
print("\n🚀 Starting Global Evaluation with CellSAM...")

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Processing Test Set"):
        sources = batch.get('dataset_source', [])
        batch_size = len(sources)

        for i in range(batch_size):
            ds_name = sources[i]
            if ds_name not in DATASET_MODALITIES:
                continue
            
            dataset_counts[ds_name] += 1
            mods = DATASET_MODALITIES[ds_name]

            # -----------------------------------
            # PREPARE & INFER H&E
            # -----------------------------------
            pred_he = None
            if mods['he_nuclei'] or mods['he_cell']:
                he_np = tensor_to_numpy_img(batch['he_image'][i])
                pred_he = safe_cellsam_predict(he_np, model, device_str)

            # -----------------------------------
            # PREPARE & INFER MIF (Only if needed)
            # -----------------------------------
            pred_mif = None
            if (mods['mif_nuclei'] or mods['mif_cell']) and 'mif_image' in batch:
                mif_np = tensor_to_numpy_img(batch['mif_image'][i])
                pred_mif = safe_cellsam_predict(mif_np, model, device_str)

            # -----------------------------------
            # CALCULATE METRICS
            # -----------------------------------
            
            # --- H&E NUCLEI ---
            if mods['he_nuclei'] and pred_he is not None:
                gt = batch['he_nuclei_instance'][i].cpu().numpy()
                if gt.max() > 0:
                    pq, dq, sq = get_fast_pq(gt, pred_he)
                    aji = aggregated_jaccard_index(gt, pred_he)
                    dataset_metrics[ds_name]['he_nuclei_pq'].append(pq)
                    dataset_metrics[ds_name]['he_nuclei_dq'].append(dq)
                    dataset_metrics[ds_name]['he_nuclei_sq'].append(sq)
                    dataset_metrics[ds_name]['he_nuclei_aji'].append(aji)

            # --- H&E CELLS ---
            if mods['he_cell'] and pred_he is not None:
                gt = batch['he_cell_instance'][i].cpu().numpy()
                if gt.max() > 0:
                    pq, dq, sq = get_fast_pq(gt, pred_he)
                    aji = aggregated_jaccard_index(gt, pred_he)
                    dataset_metrics[ds_name]['he_cell_pq'].append(pq)
                    dataset_metrics[ds_name]['he_cell_dq'].append(dq)
                    dataset_metrics[ds_name]['he_cell_sq'].append(sq)
                    dataset_metrics[ds_name]['he_cell_aji'].append(aji)

            # --- MIF NUCLEI ---
            if mods['mif_nuclei'] and pred_mif is not None:
                gt = batch['mif_nuclei_instance'][i].cpu().numpy()
                if gt.max() > 0:
                    pq, dq, sq = get_fast_pq(gt, pred_mif)
                    aji = aggregated_jaccard_index(gt, pred_mif)
                    dataset_metrics[ds_name]['mif_nuclei_pq'].append(pq)
                    dataset_metrics[ds_name]['mif_nuclei_dq'].append(dq)
                    dataset_metrics[ds_name]['mif_nuclei_sq'].append(sq)
                    dataset_metrics[ds_name]['mif_nuclei_aji'].append(aji)

            # --- MIF CELLS ---
            if mods['mif_cell'] and pred_mif is not None:
                gt = batch['mif_cell_instance'][i].cpu().numpy()
                if gt.max() > 0:
                    pq, dq, sq = get_fast_pq(gt, pred_mif)
                    aji = aggregated_jaccard_index(gt, pred_mif)
                    dataset_metrics[ds_name]['mif_cell_pq'].append(pq)
                    dataset_metrics[ds_name]['mif_cell_dq'].append(dq)
                    dataset_metrics[ds_name]['mif_cell_sq'].append(sq)
                    dataset_metrics[ds_name]['mif_cell_aji'].append(aji)

# ==========================================
# 4. PRINT TABLES FOR ALL DATASETS
# ==========================================
print("\n" + "="*80)
print("📊 FINAL EVALUATION REPORT (CellSAM) - PER DATASET")
print("="*80)

sorted_datasets = sorted(dataset_metrics.keys())

for ds_name in sorted_datasets:
    count = dataset_counts[ds_name]
    metrics = dataset_metrics[ds_name]
    
    print(f"\n📁 DATASET: {ds_name.upper()} (Tiles: {count})")
    print(f"{'Modality':<20} | {'PQ':<15} | {'DQ':<15} | {'SQ':<15} | {'AJI':<15}")
    print("-" * 90)
    
    check_list = [
        ('H&E Nuclei', 'he_nuclei'),
        ('H&E Cells',  'he_cell'),
        ('MIF Nuclei', 'mif_nuclei'),
        ('MIF Cells',  'mif_cell')
    ]
    
    for label, key in check_list:
        pq_list = metrics.get(f'{key}_pq', [])
        
        if len(pq_list) > 0:
            pq_mean = np.mean(pq_list)
            pq_std = np.std(pq_list)
            dq_mean = np.mean(metrics[f'{key}_dq'])
            dq_std = np.std(metrics[f'{key}_dq'])
            sq_mean = np.mean(metrics[f'{key}_sq'])
            sq_std = np.std(metrics[f'{key}_sq'])
            aji_mean = np.mean(metrics[f'{key}_aji'])
            aji_std = np.std(metrics[f'{key}_aji'])
            
            print(f"{label:<20} | {pq_mean:.4f}±{pq_std:.4f} | {dq_mean:.4f}±{dq_std:.4f} | {sq_mean:.4f}±{sq_std:.4f} | {aji_mean:.4f}±{aji_std:.4f}")
        else:
            status = "N/A"
            print(f"{label:<20} | {status:<15} | {status:<15} | {status:<15} | {status:<15}")
    
    print("-" * 90)

print("\n✅ All datasets processed with CellSAM.")

INFO:root:Checking for cached data
INFO:root:Making request to server


Using device: cuda

📦 Loading CellSAM model...


INFO:root:Downloading models/cellsam-models_v1.2.tar.gz with size 1.7 GB to /home/yshokrollahi/.deepcell/models
1.70GB [00:36, 50.1MB/s]                            
INFO:root:🎉 Successfully downloaded file to /home/yshokrollahi/.deepcell/models/cellsam-models_v1.2.tar.gz
INFO:root:Extracting /home/yshokrollahi/.deepcell/models/cellsam-models_v1.2.tar.gz
INFO:root:Successfully extracted /home/yshokrollahi/.deepcell/models/cellsam-models_v1.2.tar.gz into /home/yshokrollahi/.deepcell/models


✅ Model loaded successfully!

🚀 Starting Global Evaluation with CellSAM...


Processing Test Set: 100%|██████████| 1737/1737 [5:07:45<00:00, 10.63s/it]  


📊 FINAL EVALUATION REPORT (CellSAM) - PER DATASET

📁 DATASET: CPM15 (Tiles: 46)
Modality             | PQ              | DQ              | SQ              | AJI            
------------------------------------------------------------------------------------------
H&E Nuclei           | 0.5852±0.1234 | 0.7646±0.1420 | 0.7603±0.0365 | 0.5958±0.1114
H&E Cells            | N/A             | N/A             | N/A             | N/A            
MIF Nuclei           | N/A             | N/A             | N/A             | N/A            
MIF Cells            | N/A             | N/A             | N/A             | N/A            
------------------------------------------------------------------------------------------

📁 DATASET: FLUORESCENCE_DSB2018 (Tiles: 53)
Modality             | PQ              | DQ              | SQ              | AJI            
------------------------------------------------------------------------------------------
H&E Nuclei           | N/A             | N/A       